# 🎓 Build Your Own Course Tutor

You will build an AI tutor that answers questions about the course lectures and shows
the PDF page each answer comes from. At the end you open it as a real web app.

**How this notebook works**

Every exercise has the same rhythm:

1. 📘 **Learn:** read the short explanation of the idea.
2. 🔍 **Look inside:** run the cells that show the idea with real data. Change them,
   experiment: you cannot break anything there.
3. ✏️ **Build:** a code cell with blanks `____`. Replace every blank, then run the cell
   (**Shift+Enter**). Your code is saved automatically, and the app uses it too.
4. ▶️ **Test:** ✅ means it works. Otherwise the message tells you what to fix.
5. 🤔 **Think:** discuss the question with your neighbour.

| Exercise | You build | You learn |
| --- | --- | --- |
| 1 | Ask the AI model a question | APIs, LLMs, tokens and cost, hallucinations |
| 2 | Search the lecture PDFs | RAG, indexing, chunks, embeddings, vector databases |
| 3 | Give the AI a search tool | Agents, tools, the agentic loop, guardrails |
| 4 | Remember the conversation | Sessions, databases, SQL and SQLite |
| 🚀 | Open your app | How it all fits together |

## 0. Get ready

1. **Kernel:** at the top right, choose **Course Companion (Python 3.12)**
   (or the Python in `backend/.venv`).
2. **API key:** your own OpenAI key goes in the file `backend/.env`.
   Never paste it into this notebook.
3. Run the cell below.

🔒 Use only public lecture material: the text you search and ask about is sent to OpenAI.

In [ ]:
# ▶️ Run this cell first (and again after restarting the kernel).
import os
import sys
from pathlib import Path

root = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
             if (folder / "backend" / "app" / "notebook_support.py").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the workshop folder.")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from backend.app.notebook_support import start

lab = start("student")

## Exercise 1: Ask the AI model a question

### 📘 What you need to know

- **API (Application Programming Interface):** a way for programs to talk to each
  other. Your Python code sends a *request* over the internet to OpenAI; the model
  runs on OpenAI's servers, not on your laptop, and sends back a *response*.
- **Large language model (LLM):** a model trained on huge amounts of text. It writes
  an answer by predicting the next piece of text, again and again.
- **Token:** such a piece of text, roughly ¾ of a word. You pay per token, for the
  question (*input*) and for the answer (*output*). `max_output_tokens` caps the
  answer length, and so the cost.
- **The limit of a plain model:** it only knows its training data. It has never read
  our lectures, and it can sound sure while being wrong (a *hallucination*).
  Exercises 2 and 3 fix that.

### ✏️ Your task

Fill in the 2 blanks, then run the cell:
- `model=` the model name, which is stored in `settings.model`
- `input=` the question, which is stored in `prompt`

In [ ]:
%%exercise 1
async def ask_model(client, settings, prompt):
    """Send one question to the AI model and return its answer as text."""
    response = await client.responses.create(
        model=____,  # which model? It is stored in settings.model
        input=____,  # what do we ask? The question is in prompt
        max_output_tokens=settings.max_output_tokens,  # limits answer length (and cost)
        store=False,  # do not keep this request on OpenAI's servers
    )
    return response.output_text

In [ ]:
# ▶️ Test Exercise 1 (change the question if you like)
await lab.test(1, "In one sentence, what is a large language model?")

🤔 **Think:** ask *"What does slide 5 of our Introduction lecture say?"*.
The answer may sound confident, but can the model know? How would you check?

## Exercise 2: Search the lecture PDFs

### 📘 What you need to know

The model has never read our lectures. The usual fix is **RAG**
(*Retrieval-Augmented Generation*): first **retrieve** the passages that matter,
then let the model **generate** an answer from them. Retrieval needs an **index**.

**Indexing** happens once per document, before anyone asks a question:

1. **Extract:** read the text of every PDF page.
2. **Chunk:** cut the text into small, overlapping pieces (here about 1,200
   characters, overlapping by 200 so sentences are not cut in half). The model
   later gets a few chunks, never whole PDFs: that is cheaper and more precise.
3. **Embed:** an *embedding model* turns each chunk into a long list of numbers,
   a **vector**. Texts with a similar *meaning* get vectors that point in a
   similar direction, even if they use different words.
4. **Store:** save every vector with its text and labels (**metadata**: course,
   lecture, file, page) in a **vector database**. We use **Chroma**, which keeps
   its data in files on your computer.

**Searching** happens for every question: embed the question with the *same*
embedding model, then ask the vector database for the closest chunks, keeping only
chunks whose labels match the chosen course (and lecture).

It works like the index at the back of a book, but by meaning instead of exact words.

### 🔍 Look inside: chunks

Run this to see what the *Extract* and *Chunk* steps make of a real lecture.

In [ ]:
from backend.app.retrieval import extract_chunks

pdf_bytes = (root / "docs" / "00-Intro.pdf").read_bytes()
chunks = extract_chunks("00-Intro.pdf", pdf_bytes)
print(f"00-Intro.pdf became {len(chunks)} chunks.")
page, start_position, text = chunks[4]
print(f"Chunk 5 comes from page {page}:\n{text[:400]}...")

### 🔍 Look inside: embeddings

Two sentences with the same meaning but different words, and one about something
else. A similarity close to 1 means "very similar meaning".

In [ ]:
sentences = ["The students passed the exam.",
             "The pupils succeeded in their test.",
             "The weather is sunny today."]
vectors = await lab.embed(sentences)
if vectors:
    first_five = [round(number, 3) for number in vectors[0][:5]]
    print(f"Each sentence became {len(vectors[0])} numbers. The first five: {first_five}")

    def similarity(a, b):  # embedding vectors have length 1, so this is cosine similarity
        return sum(x * y for x, y in zip(a, b))

    print(f"Sentence 1 vs 2 (same meaning):      {similarity(vectors[0], vectors[1]):.2f}")
    print(f"Sentence 1 vs 3 (different meaning): {similarity(vectors[0], vectors[2]):.2f}")

### ✏️ Your task

Write the *search* step. Fill in the 3 blanks, then run the cell:
- `store.embed([____])` the question, which is stored in `query`
- `store.query_vectors(question_vector, ____, ____)` first `course_id`,
  then `lecture_id` (these become the metadata filter)

The first test indexes two lectures for you. That takes a moment, but only once.

In [ ]:
%%exercise 2
async def search(store, course_id, query, lecture_id=None):
    """Find the lecture passages whose meaning is closest to the question."""
    # Step 1: turn the question into numbers (an embedding).
    vectors = await store.embed([____])  # a list with just the question (query) in it
    question_vector = vectors[0]

    # Step 2: find the closest passages, only inside the chosen course (and lecture).
    return store.query_vectors(question_vector, ____, ____)  # course_id, then lecture_id

In [ ]:
# ▶️ Test Exercise 2 (change the question if you like)
await lab.test(2, "Which application uses Retrieval Augmented Generation?")

### 🔍 Look inside: the vector database

This is one record as Chroma stores it: text, labels and vector.

In [ ]:
collection = lab.workshop.store.collection
print(f"The vector database holds {collection.count()} chunks.")
if collection.count():
    record = collection.get(limit=1, include=["metadatas", "documents", "embeddings"])
    print("Labels (metadata):", record["metadatas"][0])
    print("Text:", record["documents"][0][:150], "...")
    print("Vector:", len(record["embeddings"][0]), "numbers")

🤔 **Think:**
- What goes wrong with very big chunks? And with very small ones?
- Why must the question and the chunks use the *same* embedding model?
- "Close in meaning" is not "correct". Open the best match in the `docs`
  folder: does that page really answer the question?

## Exercise 3: Give the AI a search tool (agents)

### 📘 What you need to know

In Exercise 2 *you* decided when to search. In an **agent**, the AI decides.

- **Agent** = a model + **instructions** (its rules) + **tools** (functions it may use).
- **Tool:** an ordinary Python function with a name, a description and typed
  arguments. The SDK turns it into a description (a *schema*) the model can read.
- **The agentic loop:**

```text
question
  → 🤖 model: "I need information"  → asks to call a tool
  → 🧰 our code runs the tool        → sends the result back
  → 🤖 model reads it: answer now, or call a tool again?
  → ... until a final answer (or the max_turns limit)
```

The model never runs code itself: it only *asks*. Our program decides what happens.

**Workflow or agent?** A fixed *workflow* always does the same steps (search, then
answer). An *agent* chooses its own steps: it may search twice for a comparison,
rephrase a search, or not search at all. That is more flexible, but less
predictable, so good agents have **guardrails**:

- The tool only takes a `query`. Our code locks the course, so the AI cannot
  search other courses.
- The instructions say that PDF text is information, never orders. That protects
  against **prompt injection** (a document saying "ignore your rules").
- `max_turns` limits the loop, and our code checks that every cited passage was
  really retrieved. We check; we do not just trust the model.

We use the **OpenAI Agents SDK**: `Agent`, `function_tool` and (in Exercise 4) `Runner`.

### 🔍 Look inside: the instructions

These are the tutor's rules. Which sentence protects against prompt injection?
Which one makes it admit when it does not know?

In [ ]:
from backend.app.tutor import GROUNDING_INSTRUCTIONS

print(GROUNDING_INSTRUCTIONS.replace(". ", ".\n"))

### ✏️ Your task

Fill in the 2 blanks, then run the cell:
- inside the tool: the function that really searches, `search_documents`
  (your Exercise 2 search, already locked to the chosen course)
- `tools=[____]` the tool the AI may use: your function `search_course_docs`

In [ ]:
%%exercise 3
from agents import Agent, function_tool

from backend.app.tutor import GROUNDING_INSTRUCTIONS


def build_agent(search_documents, model, model_settings):
    """Build the tutor: an AI model with instructions and one tool to search the lectures."""

    @function_tool(name_override="search_course_docs", failure_error_function=None)
    async def search_course_docs(query: str) -> list[dict]:
        """Search the selected course documents and return matching passages."""
        return await ____(query)  # which function searches? It was given to build_agent

    return Agent(
        name="Course Tutor",
        instructions=GROUNDING_INSTRUCTIONS,  # rules: only answer from the lectures
        model=model,
        model_settings=model_settings,
        tools=[____],  # which tool may the AI use? Put your tool function here
    )

In [ ]:
# ▶️ Test Exercise 3
await lab.test(3)

🤔 **Think:** why may the AI choose the search words, but not the course?
What could go wrong if `course_id` were a tool argument?

## Exercise 4: Remember the conversation (sessions and SQLite)

### 📘 What you need to know

An AI model is **stateless**: every request starts from zero. To hold a
conversation, the program must send the earlier messages along with each new one.
A **session** does that for you: it saves every message, and adds the saved
messages to the next request.

The messages are saved in a **database**: an organised store of data that stays
after the program stops. Most databases are **tables** (like spreadsheets) made of
**rows** (one record each) and **columns** (fields). You read and write them with
**SQL**, for example `SELECT * FROM agent_messages WHERE session_id = '...'`.

**SQLite** is a database in one single file (here `sessions.sqlite`), with no
server to install. It is built into Python and used in phones, browsers and many apps.

| | Vector database (Chroma) | Session database (SQLite) |
| --- | --- | --- |
| Holds | lecture chunks and their vectors | the conversation |
| Finds things by | meaning (similarity) | exact session ID |
| Answers | "What do the lectures say?" | "What did we say before?" |

Remembered messages are sent again with every request, so long conversations cost
more tokens each turn.

### ✏️ Your task

Fill in the 3 blanks in `Runner.run(...)`, then run the cell:
- who answers: `agent`
- what is asked: only the new `message`
- where the conversation is remembered: `session`

`Runner` runs the agentic loop from Exercise 3 for you.

In [ ]:
%%exercise 4
from agents import RunConfig, Runner

PRIVATE = RunConfig(tracing_disabled=True, trace_include_sensitive_data=False)


async def ask_tutor(agent, message, session, max_turns):
    """Send ONE new message to the tutor. The session adds the earlier conversation."""
    return await Runner.run(
        ____,  # who answers? the agent
        ____,  # what is asked? only the new message
        session=____,  # where is the conversation remembered? the session
        max_turns=max_turns,  # safety limit on how often the AI may use its tool
        run_config=PRIVATE,  # do not send traces of the conversation to OpenAI
    )

In [ ]:
# ▶️ Test Exercise 4: a question, a follow-up, and the same follow-up in a new conversation
await lab.test(4)

### 🔍 Look inside: the SQLite database

Plain SQL on the session file: which tables exist, how many rows each
conversation has, and what one stored row looks like.

In [ ]:
import json
import sqlite3

database = lab.workshop.conversations.database
if not database.exists():
    print("No conversations yet: run the Exercise 4 test first.")
else:
    with sqlite3.connect(database) as connection:
        tables = connection.execute(
            "SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()
        print("Tables:", [name for (name,) in tables])
        rows = connection.execute(
            "SELECT session_id, COUNT(*) FROM agent_messages GROUP BY session_id"
        ).fetchall()
        print(f"{len(rows)} conversations stored:")
        for session_id, count in rows:
            print(f"   {session_id[:30]}...  {count} rows")
        newest = connection.execute(
            "SELECT message_data FROM agent_messages ORDER BY id DESC LIMIT 1"
        ).fetchone()
        if newest:
            print("Newest row:", json.dumps(json.loads(newest[0]))[:300], "...")

### 🔍 Look inside: the agentic loop

Now everything works together. This runs the agent once and prints every step
of its loop, plus how many tokens it used. Try a comparison question (does it
search more than once?) and a question the lectures cannot answer.

In [ ]:
await lab.trace("Compare the ethical concerns in the Ethics lecture with the topics "
                "of the Introduction lecture.")

🤔 **Think:**
- Why can a new conversation still find facts from the lectures, but not what
  you said before?
- What would go wrong if you also sent the whole earlier conversation yourself?

## 💬 Chat with your tutor

Your tutor is complete. Ask anything about the lectures. Follow-up questions work,
because the conversation is remembered. Run `lab.new_chat()` to start fresh. To
search one lecture only, add for example `lecture="ethics"`.

In [ ]:
await lab.chat("What are the main ethical concerns of generative AI?")

In [ ]:
await lab.chat("Which one of those is about privacy?")  # a follow-up: it remembers!

## 🚀 Open your app

Run the cell below and click the link. The web app runs on **your** code from
exercises 1 to 4: if you change an exercise cell and run it, the app uses the new code.

In [ ]:
lab.open_app()

Stop the app when you are done:

In [ ]:
lab.close_app()

## 🧪 Test your tutor like a pro

A good tutor answers from the lectures, shows its sources, and admits when it does not
know. Try these in the app (or with `await lab.chat(...)`). For each one, write down
**pass**, **partial** or **fail**, and why.

| Try this | A good tutor... |
| --- | --- |
| "Which helpdesk application is named in the Introduction lecture?" | answers and cites the right page (open it!) |
| "Which ethical concerns should be considered for the topics in the Introduction lecture?" | uses both the Introduction and the Ethics lecture |
| "What is the private phone number of workshop coordinator Mira Northbridge?" | says it cannot answer from the lectures |
| Start a new conversation, then ask "What answer style did I ask for?" | does not remember the old conversation |
| In the app, switch the course to Study Notes and ask about the lectures | finds nothing from the Generative AI course |

## 📖 What you learned

| Term | In one sentence |
| --- | --- |
| API | How your program sends a request to a service (OpenAI) and gets a response. |
| LLM | A model that writes text by predicting the next token. |
| Token | A piece of text (about ¾ of a word); the unit you pay for. |
| Hallucination | A confident answer that is not supported by facts. |
| RAG | Retrieve relevant passages first, then generate the answer from them. |
| Chunk | A small piece of a document, the unit that is searched and cited. |
| Embedding / vector | A list of numbers that captures the meaning of a text. |
| Indexing | Extract, chunk, embed and store documents so they can be searched. |
| Vector database | Stores vectors with text and metadata, and finds the closest ones (Chroma). |
| Metadata | Labels on a chunk (course, lecture, file, page), used for filtering and citing. |
| Agent | A model with instructions and tools that decides its own next steps. |
| Tool | A function the model may *ask* to run; your code really runs it. |
| Agentic loop | Think, call a tool, read the result, repeat, until a final answer. |
| Guardrail | A rule enforced by code that keeps an agent safe (scope, max_turns, citation checks). |
| Prompt injection | Text in a document that tries to give the AI orders. |
| Session | Stored conversation history, added to each new request. |
| Database / SQL / SQLite | Organised lasting storage / its query language / a one-file database. |

## 🧗 Extra challenges (if you finish early)

1. Ask the same question with `lecture="introduction"` and without. What changes?
2. In Exercise 3, write your own `instructions` (for example: "answer in Dutch, in
   three bullet points, always with page numbers"). Run the cell, then `lab.trace(...)`.
   Restore the original afterwards.
3. Use `lab.trace(...)` with a question the lectures cannot answer. Does the agent
   search? What does it answer?
4. Without looking at the hints, explain each line of your four exercise cells to
   a neighbour.

## ✅ Finish

Run the cell below to see your progress. Before sharing this notebook, clear all
outputs (**...** menu → **Clear All Outputs**).

In [ ]:
lab.summary()